# Tahap 2: Case Representation
## Alur Kerja Tahap 2:
1. **Ekstraksi Metadata**: nomor perkara, tanggal, jenis perkara, pasal, pihak (Terdakwa)
2. **Ekstraksi Konten Kunci**: ringkasan fakta (barang bukti & dakwaan), argumen hukum utama (amar & pasal yang diputuskan)
3. **Feature Engineering**: panjang teks (jumlah kata), bag-of-words sederhana
4. **Penyimpanan**: ke `data/processed/cases.csv` dengan kolom `case_id, no_perkara, tanggal, ringkasan_fakta, pasal, pihak, text_full` (sesuai contoh tabel di ketentuan tugas), ditambah beberapa kolom pendukung



## 0. Setup & Path

In [ ]:
import re
import json
from pathlib import Path

import pandas as pd

BASE_DIR = Path(".").resolve().parent  

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw dir       : {RAW_DIR}")
print(f"Processed dir : {PROCESSED_DIR}")
print(f"Raw dir exists?  {RAW_DIR.exists()}")


Raw dir       : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\raw
Processed dir : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\processed
Raw dir exists?  True


In [ ]:
mapping_path = RAW_DIR / "file_mapping.csv"
file_mapping_df = pd.read_csv(mapping_path)
print(f"Jumlah dokumen pada mapping: {len(file_mapping_df)}")
file_mapping_df.head()


Jumlah dokumen pada mapping: 30


,case_id,original_filename,extraction_method,char_count
0,case_001,putusan_1002_pid.sus_2021_pn_jkt.tim_202606202...,pymupdf,146723
1,case_002,putusan_1037_pid.sus_2021_pn_jkt.tim_202606202...,pymupdf,35683
2,case_003,putusan_11_pid.sus_2022_pn_jkt.tim_20260620225...,pymupdf,104992
3,case_004,putusan_202_pid.sus_2023_pn_jkt.tim_2026061911...,pymupdf,497686
4,case_005,putusan_203_pid.sus_2023_pn_jkt.tim_2026062022...,pymupdf,514618


In [ ]:
case_texts = {}
txt_files = sorted(RAW_DIR.glob("case_*.txt"))

for f in txt_files:
    case_texts[f.stem] = f.read_text(encoding="utf-8")

print(f"Jumlah dokumen teks dimuat: {len(case_texts)}")
assert len(case_texts) >= 30, "Jumlah dokumen kurang dari 30 - cek kembali hasil Tahap 1!"


Jumlah dokumen teks dimuat: 30


## 1. Ekstraksi Metadata

Metadata yang diekstrak (sesuai ketentuan tugas): **nomor perkara**, **tanggal putusan**, **jenis perkara**,
**pasal**, dan **pihak (Terdakwa)**.

Pola regex di bawah ini disusun berdasarkan inspeksi manual terhadap beberapa sampel nyata dari dataset
(bukan asumsi generik), termasuk variasi format hasil OCR. Karena variasi format antar dokumen cukup besar,
setiap fungsi ekstraksi mencoba **beberapa pola alternatif** secara berurutan dan mengembalikan `None`
jika semuanya gagal — bukan melempar error.


In [ ]:
def extract_nomor_perkara(text: str) -> str | None:
    """
    Ekstrak nomor perkara, contoh target: "463/Pid.Sus/2024/PN Jkt.Tim".
    Diambil dari judul resmi "P U T U S A N" / "PUTUSAN" diikuti baris "Nomor ...".
    """
    patterns = [
        r"P\s*U\s*T\s*U\s*S\s*A\s*N\s*\n+\s*Nomor\s*:?\s*(\d+/Pid\.?\s*Sus/\d{4}/PN\.?\s*Jkt\.?\s*Tim\.?)",
        r"NOMOR\s*:?\s*(\d+/PID\.?\s*SUS/\d{4}/PN\.?\s*JKT\.?\s*TIM\.?)",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return re.sub(r"\s+", " ", m.group(1)).strip()
    return None


def extract_tanggal_putusan(text: str) -> str | None:
    """
    Ekstrak tanggal putusan DIUCAPKAN (bukan tanggal sidang/penahanan), contoh target: "04 April 2022".
    Diambil dari kalimat baku "...diucapkan dalam sidang terbuka untuk umum hari ..., Tanggal ...".

    Variasi yang ditemukan: sebagian dokumen menulis "diucapkan ... pada hari dan tanggal itu juga"
    (merujuk balik ke tanggal pada kalimat "Demikianlah diputuskan ... pada hari ..., tanggal ..."
    sebelumnya) - bukan menyebut tanggal baru secara eksplisit. Pattern terakhir menangani kasus ini.
    """
    patterns = [
        r"diucapkan\s+dalam\s+sidang\s+terbuka\s+untuk\s+umum\s+hari\s+\w+,?\s*"
        r"[Tt]anggal\s+(\d{1,2}\s+\w+\s+\d{4})",
        r"diucapkan\b.{0,80}?[Tt]anggal\s+(\d{1,2}\s+\w+\s+\d{4})",
        r"diputuskan\b.{0,100}?[Tt]anggal\s+(\d{1,2}\s+\w+\s+\d{4}).{0,200}?"
        r"diucapkan\b.{0,60}?hari\s+dan\s+tanggal\s+itu\s+juga",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            return re.sub(r"\s+", " ", m.group(1)).strip()
    return None


def extract_jenis_perkara(nomor_perkara: str | None) -> str | None:
    """
    Tentukan jenis perkara dari kode dalam nomor perkara. Untuk dataset ini, semua dokumen
    seharusnya berjenis "Pid.Sus" (Pidana Khusus - UU ITE), tapi tetap dideteksi otomatis
    dari teks nomor perkara agar konsisten dan dapat diverifikasi (bukan di-hardcode).
    """
    if not nomor_perkara:
        return None
    m = re.search(r"Pid\.?\s*Sus", nomor_perkara, flags=re.IGNORECASE)
    return "Pidana Khusus (Pid.Sus)" if m else None


def extract_nama_terdakwa(text: str) -> str | None:
    """
    Ekstrak nama Terdakwa dari blok identitas. Menangani 2 variasi format yang ditemukan:
    1. Format rapat: "Nama Lengkap : NAMA Bin AYAH." dalam satu baris/dekat
    2. Format multi-baris (umum di hasil OCR/bullet): "Nama Lengkap\n:\nNAMA;"
    Untuk perkara dengan multiple terdakwa, hanya terdakwa PERTAMA yang diambil (representasi utama kasus).
    """
    patterns = [
        r"Nama\s+[Ll]engkap\s*\n?\s*:\s*\n?\s*([A-Z][A-Za-z\.\s]+?)(?:[;\n.]|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            nama = m.group(1).strip()
            return nama.strip(" .;:")
    return None


def extract_pasal(text: str) -> list[str]:
    """
    Ekstrak SEMUA pasal yang disebut dalam konteks dakwaan/putusan (untuk referensi top-N nanti).
    Mengembalikan list pasal unik (urutan kemunculan), dibatasi 10 teratas agar tidak membengkak.
    """
    matches = re.findall(
        r"Pasal\s+\d+(?:\s+ayat\s*\(\d+\))?(?:\s+(?:Jo\.?|jo\.?)\s+Pasal\s+\d+(?:\s+ayat\s*\(\d+\))?)*",
        text,
    )
    seen = []
    for m in matches:
        norm = re.sub(r"\s+", " ", m).strip()
        if norm not in seen:
            seen.append(norm)
    return seen[:10]


def extract_lama_hukuman_bulan(text: str) -> int | None:
    """
    Ekstrak lama hukuman PENJARA dari bagian amar (MENGADILI) yang SESUNGGUHNYA, dikonversi ke
    total BULAN supaya mudah dibandingkan antar kasus (1 tahun = 12 bulan).

    CATATAN PENTING dari inspeksi manual dataset: pada beberapa dokumen, kata "MENGADILI" atau
    struktur kalimat bernumbing serupa amar ("1. Menyatakan...2. Menjatuhkan...") muncul LEBIH
    DARI SEKALI - kemunculan pertama seringkali adalah RINGKASAN TUNTUTAN JAKSA (requisitoir)
    yang dikutip dalam pertimbangan hakim, BUKAN vonis akhir. Vonis yang sesungguhnya berada pada
    kemunculan "MENGADILI" PALING AKHIR dalam dokumen. Fungsi ini SELALU mengambil kemunculan
    terakhir untuk menghindari salah ambil angka tuntutan jaksa sebagai vonis.

    Mengembalikan None jika tidak ditemukan durasi penjara - termasuk untuk kasus VONIS BEBAS/LEPAS
    (lihat extract_status_vonis) yang memang tidak punya durasi penjara karena Terdakwa dibebaskan.

    PENTING - ini bantuan EKSTRAKSI OTOMATIS untuk draft awal label, BUKAN hasil final.
    SELALU cek ulang ke teks asli (kolom argumen_hukum / text_full) sebelum dipakai sebagai label resmi.
    """
    matches = list(re.finditer(r"M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?", text, flags=re.IGNORECASE))
    if not matches:
        return None
    amar_text = text[matches[-1].end():matches[-1].end() + 2000]  

    pattern = (
        r"selama\s+(\d+)\s*\(\s*[\w\s]+?\s*\)\s*[Tt]ahun"
        r"(?:\s+(?:dan\s+)?(\d+)\s*\(\s*[\w\s]+?\s*\)\s*[Bb]ulan)?"
    )
    m = re.search(pattern, amar_text, flags=re.IGNORECASE)
    if not m:
        return None

    tahun = int(m.group(1)) if m.group(1) else 0
    bulan = int(m.group(2)) if m.group(2) else 0
    return tahun * 12 + bulan


def extract_status_vonis(text: str) -> str:
    """
    Deteksi status vonis dasar: "Bersalah" atau "Bebas/Lepas", diambil dari amar (MENGADILI)
    kemunculan TERAKHIR (lihat catatan di extract_lama_hukuman_bulan tentang kemunculan ganda).
    Berguna sebagai konteks tambahan saat lama_hukuman_bulan_draft kosong - membedakan antara
    "gagal diekstrak karena formatnya tidak terduga" vs "memang tidak ada hukuman karena divonis bebas".
    """
    matches = list(re.finditer(r"M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?", text, flags=re.IGNORECASE))
    if not matches:
        return "Tidak diketahui"
    amar_text = text[matches[-1].end():matches[-1].end() + 500]

    if re.search(r"tidak\s+terbukti|membebaskan|bebas\s+dari\s+segala\s+dakwaan|lepas\s+dari\s+segala\s+tuntutan", amar_text, flags=re.IGNORECASE):
        return "Bebas/Lepas"
    if re.search(r"terbukti\s+(?:secara\s+sah\s+dan\s+meyakinkan\s+)?bersalah", amar_text, flags=re.IGNORECASE):
        return "Bersalah"
    return "Tidak diketahui"


def extract_pasal_terbukti(text: str, pasal_list: list[str]) -> str | None:
    """
    Estimasi pasal UTAMA yang terbukti (untuk draft label), diambil dari pasal PERTAMA yang
    disebut dalam 800 karakter SETELAH kemunculan TERAKHIR "MENGADILI" (vonis sesungguhnya,
    bukan ringkasan tuntutan jaksa yang mungkin disebut lebih dulu - lihat catatan di
    extract_lama_hukuman_bulan). Jika tidak ditemukan di bagian amar, fallback ke pasal PERTAMA
    dari `pasal_list` (hasil extract_pasal di seluruh dokumen) sebagai estimasi kasar.

    PENTING - ini bantuan EKSTRAKSI OTOMATIS untuk draft awal label, BUKAN hasil final.
    Pasal yang "disebut pertama" tidak selalu sama dengan pasal yang benar-benar dinyatakan
    terbukti oleh hakim, terutama untuk dakwaan dengan banyak pasal alternatif/kumulatif.
    SELALU verifikasi ke bagian amar putusan asli sebelum dipakai sebagai label resmi.
    """
    matches = list(re.finditer(r"M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?", text, flags=re.IGNORECASE))
    if matches:
        amar_text = text[matches[-1].end():matches[-1].end() + 800]
        m_pasal = re.search(r"Pasal\s+\d+(?:\s+ayat\s*\(\d+\))?", amar_text, flags=re.IGNORECASE)
        if m_pasal:
            return re.sub(r"\s+", " ", m_pasal.group()).strip()

    return pasal_list[0] if pasal_list else None


In [ ]:
metadata_records = []

for case_id, text in case_texts.items():
    nomor_perkara = extract_nomor_perkara(text)
    tanggal = extract_tanggal_putusan(text)
    jenis_perkara = extract_jenis_perkara(nomor_perkara)
    nama_terdakwa = extract_nama_terdakwa(text)
    pasal_list = extract_pasal(text)
    status_vonis = extract_status_vonis(text)
    lama_hukuman_bulan = extract_lama_hukuman_bulan(text)
    pasal_terbukti = extract_pasal_terbukti(text, pasal_list)

    metadata_records.append({
        "case_id": case_id,
        "no_perkara": nomor_perkara,
        "tanggal": tanggal,
        "jenis_perkara": jenis_perkara,
        "pihak": nama_terdakwa,  
        "pasal": "; ".join(pasal_list) if pasal_list else None,
        "jumlah_pasal_disebut": len(pasal_list),
        "status_vonis_draft": status_vonis,
        "pasal_terbukti_draft": pasal_terbukti,
        "lama_hukuman_bulan_draft": lama_hukuman_bulan,
    })

metadata_df = pd.DataFrame(metadata_records).sort_values("case_id").reset_index(drop=True)
metadata_df.head(10)


,case_id,no_perkara,tanggal,jenis_perkara,pihak,pasal,jumlah_pasal_disebut,status_vonis_draft,pasal_terbukti_draft,lama_hukuman_bulan_draft
0,case_001,1002/Pid.Sus/2022/PN Jkt.Tim,04 APRIL 2022,Pidana Khusus (Pid.Sus),NOVALDI PRATOMO,Pasal 30 ayat (1) jo Pasal 46; Pasal 64 ayat (...,10,Bersalah,Pasal 30 ayat (1) jo Pasal 46,66.0
1,case_002,1037/PID.SUS/2021/PN.JKT.Tim.,22 Maret 2022,Pidana Khusus (Pid.Sus),A.H,Pasal 51 ayat (1); Pasal 51 ayat (1) Jo Pasal ...,4,Bersalah,Pasal 51 ayat (1),15.0
2,case_003,11/Pid.Sus/2022/PN Jkt.Tim,29 Maret 2022,Pidana Khusus (Pid.Sus),Jabbar,Pasal 45; Pasal 28 ayat (1); Pasal 55 ayat (1)...,6,Bersalah,Pasal 45,36.0
3,case_004,202/Pid.Sus/2023/PN. Jkt Tim,8 Januari 2024,Pidana Khusus (Pid.Sus),HARIS AZHAR,Pasal 27 ayat (3) jo. Pasal 45 ayat (3); Pasal...,10,Bebas/Lepas,Pasal 27 ayat (3) jo. Pasal 45 ayat (3),NaN
4,case_005,203/Pid.Sus/2023/PN JKT.TIM,8 Januari 2024,Pidana Khusus (Pid.Sus),FATIAH MAULIDIYANTY,Pasal 27 ayat (3) jo. Pasal 45 ayat (3); Pasal...,10,Bebas/Lepas,Pasal 27 ayat (3) jo. Pasal 45 ayat (3),NaN
5,case_006,212/Pid.Sus/2025/PN Jkt.Tim,None,Pidana Khusus (Pid.Sus),TERDAKWA,Pasal 45 ayat (1) jo Pasal 27 ayat (1); Pasal ...,8,Bersalah,Pasal 45 ayat (1) jo Pasal 27 ayat (1),24.0
6,case_007,330/Pid.Sus/2025/PN Jkt.Tim,14 Agustus 2025,Pidana Khusus (Pid.Sus),Eko Candra Setiawan,Pasal 28 ayat (1) Jo. Pasal 45; Pasal 55 ayat ...,7,Bersalah,Pasal 28 ayat (1) Jo. Pasal 45,36.0
7,case_008,351/Pid.Sus/2025/PN Jkt. Tim,None,Pidana Khusus (Pid.Sus),IWAN MARDAN AKBAR BIN ALM,Pasal 84 ayat (2); Pasal 45 ayat (1) jo. Pasal...,6,Bersalah,Pasal 84 ayat (2),36.0
8,case_009,365/Pid.Sus/2023/PN JKT.Tim,19 September 2023,Pidana Khusus (Pid.Sus),TAYO TOLU OMODARATAN alias ADAM,Pasal 2; Pasal 378 jo Pasal 55; Pasal 3; Pasal...,10,Bersalah,Pasal 2,48.0
9,case_010,463/Pid.Sus/2024/PN Jkt.Tim,19 Desember 2024,Pidana Khusus (Pid.Sus),KARMANSYAH LILI,Pasal 2 ayat (1); Pasal 32 ayat (2) jo Pasal 3...,10,Bersalah,Pasal 2 ayat (1),30.0


> **Catatan tentang kolom "pihak":** ketentuan tugas mencontohkan kolom ini untuk perkara **perdata**
> ("Penggugat & Tergugat", contoh: "A vs. B"). Karena domain dataset ini adalah **pidana** (UU ITE), pihak
> yang relevan adalah **Terdakwa** (tidak ada Penggugat/Tergugat dalam perkara pidana). Kolom `pihak` di sini
> diisi dengan nama Terdakwa utama tiap perkara.


In [ ]:
print("=== Tingkat keberhasilan ekstraksi metadata ===")
for col in ["no_perkara", "tanggal", "jenis_perkara", "pihak", "pasal"]:
    n_filled = metadata_df[col].notna().sum()
    n_total = len(metadata_df)
    print(f"  {col:<15}: {n_filled}/{n_total} terisi ({n_filled/n_total:.0%})")

missing_any = metadata_df[
    metadata_df[["no_perkara", "tanggal", "pihak"]].isna().any(axis=1)
]
if len(missing_any) > 0:
    print(f"\n⚠️  {len(missing_any)} dokumen punya minimal 1 field metadata kosong - perlu dicek manual:")
    print(missing_any[["case_id", "no_perkara", "tanggal", "pihak"]].to_string(index=False))
else:
    print("\n✅ Semua dokumen berhasil diekstrak metadatanya secara lengkap.")


=== Tingkat keberhasilan ekstraksi metadata ===
  no_perkara     : 25/30 terisi (83%)
  tanggal        : 20/30 terisi (67%)
  jenis_perkara  : 25/30 terisi (83%)
  pihak          : 28/30 terisi (93%)
  pasal          : 30/30 terisi (100%)

⚠️  14 dokumen punya minimal 1 field metadata kosong - perlu dicek manual:
 case_id                   no_perkara          tanggal                           pihak
case_006  212/Pid.Sus/2025/PN Jkt.Tim             None                        TERDAKWA
case_008 351/Pid.Sus/2025/PN Jkt. Tim             None       IWAN MARDAN AKBAR BIN ALM
case_012  465/Pid.Sus/2024/PN Jkt.Tim             None               HABIB WIKADIPUTRA
case_013                         None             None                           XXXXX
case_016  525/Pid.Sus/2024/PN Jkt.Tim 19 Desember 2024                            None
case_017  565/Pid.Sus/2025/PN Jkt.Tim             None                           ROBIN
case_018  566/Pid.Sus/2025/PN Jkt.Tim             None           OCHEREND RA

## 2. Ekstraksi Konten Kunci

Sesuai ketentuan tugas, dua jenis konten kunci yang diekstrak:
1. **Ringkasan fakta** (barang bukti & dakwaan) — diambil dari bagian dakwaan, dimulai dari kalimat baku
   "Bahwa ia terdakwa..." yang umum pada dakwaan pidana.
2. **Argumen hukum utama** (amar putusan & pasal yang diputuskan) — diambil dari bagian "MENGADILI"
   (toleran spasi antar huruf seperti "M E N G A D I L I" yang umum pada dokumen ini).

Pendekatan ini adalah **ekstraksi bagian relevan** dari teks asli (bukan ringkasan yang di-paraphrase ulang
oleh model bahasa) — cukup memadai untuk tahap retrieval berbasis kemiripan teks (TF-IDF/embedding) di
Tahap 3, sekaligus tetap dapat ditelusuri kembali ke teks aslinya.


In [ ]:
def extract_ringkasan_fakta(text: str, max_chars: int = 2000) -> str | None:
    """
    Ekstrak potongan teks dakwaan sebagai representasi "ringkasan fakta", dimulai dari kalimat
    baku pembuka dakwaan pidana. Menangani beberapa variasi gaya penulisan yang ditemukan di dataset:
    - "Bahwa ia terdakwa..." (1 terdakwa, gaya umum)
    - "Bahwa mereka Terdakwa I .../Bahwa awalnya Terdakwa I ..." (2+ terdakwa)
    - "Bahwa Terdakwa NAMA NAMA..." (nama ALL CAPS disebut langsung tanpa kata penghubung)
    - Variasi dengan whitespace tidak standar antar kata (umum pada hasil ekstraksi PDF tertentu)

    PENTING: pattern nama ALL CAPS (mis. "[A-Z]{2,}") TIDAK boleh dicek dengan flag re.IGNORECASE,
    karena IGNORECASE membuat [A-Z] ikut cocok huruf kecil juga, sehingga bisa salah menangkap kata
    biasa seperti "memohon" sebagai jika itu nama. Pattern ini dicek case-sensitive secara terpisah.
    """
    case_insensitive_patterns = [
        r"Bahwa\s+ia\s+terdakwa\b",
        r"Bahwa\s+mereka\s+[Tt]erdakwa\s+I\b",       
        r"Bahwa\s+awalnya\s+[Tt]erdakwa\s+I\b",      
        r"bahwa\s+terdakwa\s+I\.?\s+[A-Za-z]+.{0,60}?dan\s+terdakwa\s+II",  
                                                            
    ]
    for pat in case_insensitive_patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return text[m.start():m.start() + max_chars].strip()

    m = re.search(r"Bahwa\s+[Tt]erdakwa\s+[A-Z]{2,}\s+[A-Z]{2,}", text)
    if m:
        return text[m.start():m.start() + max_chars].strip()

    m = re.search(r"Bahwa\s+(?:para\s+)?[Tt]erdakwa\b.{0,40}?pada\s+hari", text, flags=re.IGNORECASE)
    if m:
        return text[m.start():m.start() + max_chars].strip()

    return None


def extract_argumen_hukum(text: str, max_chars: int = 1500) -> str | None:
    """
    Ekstrak bagian amar putusan (argumen hukum utama: vonis & pasal yang diputuskan),
    dimulai dari kata kunci "MENGADILI" (toleran spasi antar huruf).
    """
    m = re.search(r"M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?", text, flags=re.IGNORECASE)
    if not m:
        return None
    start = m.end()
    snippet = text[start:start + max_chars].strip()
    return snippet if snippet else None


In [24]:
for record in metadata_records:
    text = case_texts[record["case_id"]]
    record["ringkasan_fakta"] = extract_ringkasan_fakta(text)
    record["argumen_hukum"] = extract_argumen_hukum(text)

metadata_df = pd.DataFrame(metadata_records).sort_values("case_id").reset_index(drop=True)

for col in ["ringkasan_fakta", "argumen_hukum"]:
    n_filled = metadata_df[col].notna().sum()
    print(f"{col:<15}: {n_filled}/{len(metadata_df)} dokumen berhasil diekstrak ({n_filled/len(metadata_df):.0%})")


ringkasan_fakta: 26/30 dokumen berhasil diekstrak (87%)
argumen_hukum  : 30/30 dokumen berhasil diekstrak (100%)


## 3. Feature Engineering

Sesuai ketentuan tugas: **hitung length (jumlah kata)** dan **bag-of-words sederhana**.

Bag-of-words di sini dibuat sebagai daftar 10 kata paling sering muncul per dokumen (setelah membuang
stopword umum Bahasa Indonesia) — representasi sederhana sesuai permintaan tugas. Representasi vektor
yang lebih lengkap (TF-IDF penuh) akan dibangun di **Tahap 3: Case Retrieval**, karena vektor TF-IDF
perlu dibangun di seluruh korpus sekaligus (bukan per dokumen).


In [ ]:
STOPWORDS_ID = {
    "yang", "dan", "di", "ke", "dari", "pada", "untuk", "dengan", "ini", "itu",
    "atau", "dalam", "adalah", "akan", "telah", "tidak", "tersebut", "oleh",
    "sebagai", "juga", "atas", "bahwa", "tidak", "dapat", "ada", "saat", "para",
    "satu", "dua", "tiga", "ia", "nya", "tahun", "hari", "yaitu", "secara",
}


def compute_bag_of_words(text: str, top_n: int = 10) -> str:
    """
    Hitung bag-of-words sederhana: top-N kata (selain stopword) yang paling sering muncul,
    dikembalikan sebagai string "kata1:jumlah1, kata2:jumlah2, ...".
    """
    words = re.findall(r"[A-Za-z]+", text.lower())
    words = [w for w in words if w not in STOPWORDS_ID and len(w) > 2]

    counts = {}
    for w in words:
        counts[w] = counts.get(w, 0) + 1

    top_words = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return ", ".join(f"{w}:{c}" for w, c in top_words)


In [26]:
for record in metadata_records:
    text = case_texts[record["case_id"]]
    record["jumlah_kata"] = len(text.split())
    record["jumlah_karakter"] = len(text)
    record["bag_of_words_top10"] = compute_bag_of_words(text)

metadata_df = pd.DataFrame(metadata_records).sort_values("case_id").reset_index(drop=True)
metadata_df[["case_id", "jumlah_kata", "bag_of_words_top10"]].head(5)


,case_id,jumlah_kata,bag_of_words_top10
0,case_001,21316,"rekening:680, puluh:493, ratus:476, nama:410, ..."
1,case_002,4989,"terdakwa:186, saksi:119, facebook:90, akun:78,..."
2,case_003,15390,"terdakwa:347, putra:247, siregar:240, idr:198,..."
3,case_004,70654,"saksi:1213, luhut:708, haris:549, azhar:512, t..."
4,case_005,73145,"saksi:1419, luhut:708, haris:528, azhar:483, t..."


## 4. Gabungkan dengan Teks Lengkap & Siapkan Kolom Label

Menambahkan kolom `text_full` (isi teks lengkap hasil Tahap 1, dibutuhkan untuk membangun representasi
vektor TF-IDF/embedding di Tahap 3) dan kolom **`label`** berisi gabungan **pasal terbukti + lama hukuman**
(sesuai keputusan label yang dipilih untuk evaluasi Tahap 5), dengan format `"<pasal> | <lama hukuman>"`.

> ⚠️ **PENTING:** kolom `label` di sini adalah **DRAFT hasil ekstraksi otomatis** dari `pasal_terbukti_draft`
> dan `lama_hukuman_bulan_draft` (lihat fungsi `extract_pasal_terbukti` dan `extract_lama_hukuman_bulan` di
> atas) - **BUKAN label final yang siap dipakai begitu saja**. Ekstraksi otomatis ini bisa salah, terutama
> untuk kasus dengan dakwaan alternatif/kumulatif kompleks atau kalimat vonis dengan format tidak biasa.
> Kolom `label_verified` ditambahkan dengan nilai `False` di semua baris sebagai **pengingat** bahwa setiap
> baris masih perlu Anda cek manual terhadap kolom `argumen_hukum` (atau `text_full`) sebelum dipakai resmi
> di Tahap 5. Setelah dicek dan dikoreksi (jika perlu) di Excel/spreadsheet, ubah nilainya jadi `True`.


In [ ]:
metadata_df["text_full"] = metadata_df["case_id"].map(case_texts)

metadata_df = metadata_df.merge(
    file_mapping_df[["case_id", "original_filename", "extraction_method"]],
    on="case_id", how="left",
)

def build_label_draft(row):
    pasal = row["pasal_terbukti_draft"] if pd.notna(row["pasal_terbukti_draft"]) else "?"
    bulan = row["lama_hukuman_bulan_draft"]
    status = row["status_vonis_draft"]

    if status == "Bebas/Lepas":
        durasi = "Bebas/Lepas"
    elif pd.notna(bulan):
        tahun_bagian = int(bulan) // 12
        bulan_bagian = int(bulan) % 12
        if tahun_bagian and bulan_bagian:
            durasi = f"{tahun_bagian} tahun {bulan_bagian} bulan"
        elif tahun_bagian:
            durasi = f"{tahun_bagian} tahun"
        else:
            durasi = f"{bulan_bagian} bulan"
    else:
        durasi = "? (cek manual - status: " + status + ")"
    return f"{pasal} | {durasi}"

metadata_df["label"] = metadata_df.apply(build_label_draft, axis=1)
metadata_df["label_verified"] = False  

column_order = [
    "case_id", "no_perkara", "tanggal", "jenis_perkara", "ringkasan_fakta", "argumen_hukum",
    "pasal", "jumlah_pasal_disebut", "pihak",
    "label", "label_verified", "status_vonis_draft", "pasal_terbukti_draft", "lama_hukuman_bulan_draft",
    "jumlah_kata", "jumlah_karakter", "bag_of_words_top10",
    "original_filename", "extraction_method", "text_full",
]
metadata_df = metadata_df[column_order]
metadata_df[["case_id", "label", "label_verified"]].head(10)


,case_id,label,label_verified
0,case_001,Pasal 30 ayat (1) jo Pasal 46 | 5 tahun 6 bulan,False
1,case_002,Pasal 51 ayat (1) | 1 tahun 3 bulan,False
2,case_003,Pasal 45 | 3 tahun,False
3,case_004,Pasal 27 ayat (3) jo. Pasal 45 ayat (3) | Beba...,False
4,case_005,Pasal 27 ayat (3) jo. Pasal 45 ayat (3) | Beba...,False
5,case_006,Pasal 45 ayat (1) jo Pasal 27 ayat (1) | 2 tahun,False
6,case_007,Pasal 28 ayat (1) Jo. Pasal 45 | 3 tahun,False
7,case_008,Pasal 84 ayat (2) | 3 tahun,False
8,case_009,Pasal 2 | 4 tahun,False
9,case_010,Pasal 2 ayat (1) | 2 tahun 6 bulan,False


In [ ]:
print("=== Distribusi status_vonis_draft ===")
print(metadata_df["status_vonis_draft"].value_counts())

print("\n=== Distribusi pasal_terbukti_draft (cek sekilas, pastikan variasinya wajar) ===")
print(metadata_df["pasal_terbukti_draft"].value_counts())

print(f"\n=== Statistik lama_hukuman_bulan_draft (hanya untuk kasus 'Bersalah') ===")
bersalah_mask = metadata_df["status_vonis_draft"] == "Bersalah"
print(metadata_df.loc[bersalah_mask, "lama_hukuman_bulan_draft"].describe())

n_missing_pasal = metadata_df["pasal_terbukti_draft"].isna().sum()
n_missing_durasi_padahal_bersalah = (bersalah_mask & metadata_df["lama_hukuman_bulan_draft"].isna()).sum()

if n_missing_pasal > 0 or n_missing_durasi_padahal_bersalah > 0:
    print(f"\n⚠️  Ada draft label yang WAJIB dicek manual (bukan kasus Bebas/Lepas yang sah kosong):")
    problem_mask = (
        metadata_df["pasal_terbukti_draft"].isna()
        | (bersalah_mask & metadata_df["lama_hukuman_bulan_draft"].isna())
    )
    incomplete = metadata_df[problem_mask]
    print(incomplete[["case_id", "status_vonis_draft", "pasal_terbukti_draft", "lama_hukuman_bulan_draft"]]
          .to_string(index=False))
else:
    print(f"\n✅ Semua draft label terisi wajar (termasuk kasus Bebas/Lepas yang sah tidak punya durasi).")


=== Distribusi status_vonis_draft ===
status_vonis_draft
Bersalah       28
Bebas/Lepas     2
Name: count, dtype: int64

=== Distribusi pasal_terbukti_draft (cek sekilas, pastikan variasinya wajar) ===
pasal_terbukti_draft
Pasal 2 ayat (1)                                      4
Pasal 27                                              3
Pasal 45                                              3
Pasal 27 ayat (3) jo. Pasal 45 ayat (3)               2
Pasal 28 ayat (1) jo. Pasal 45                        2
Pasal 45 ayat (1) jo Pasal 27 ayat (1)                1
Pasal 51 ayat (1)                                     1
Pasal 30 ayat (1) jo Pasal 46                         1
Pasal 84 ayat (2)                                     1
Pasal 2                                               1
Pasal 480                                             1
Pasal 45 ayat (4) Jo Pasal 27 ayat (4)                1
Pasal 28 ayat (1) Jo. Pasal 45                        1
Pasal 36 Jo Pasal 30 ayat (3)                     

## 5. Simpan Output

Disimpan ke `data/processed/cases.csv` sesuai ketentuan tugas. Karena kolom `text_full`, `ringkasan_fakta`,
dan `argumen_hukum` bisa mengandung newline/koma yang kompleks, CSV disimpan dengan quoting penuh agar
aman dibuka kembali (termasuk di Excel).


In [ ]:
output_path = PROCESSED_DIR / "cases.csv"
metadata_df.to_csv(output_path, index=False, quoting=1)  

print(f"✅ Tersimpan: {output_path}")
print(f"   Jumlah baris : {len(metadata_df)}")
print(f"   Jumlah kolom : {len(metadata_df.columns)}")
print(f"   Kolom        : {list(metadata_df.columns)}")


✅ Tersimpan: C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\processed\cases.csv
   Jumlah baris : 30
   Jumlah kolom : 20
   Kolom        : ['case_id', 'no_perkara', 'tanggal', 'jenis_perkara', 'ringkasan_fakta', 'argumen_hukum', 'pasal', 'jumlah_pasal_disebut', 'pihak', 'label', 'label_verified', 'status_vonis_draft', 'pasal_terbukti_draft', 'lama_hukuman_bulan_draft', 'jumlah_kata', 'jumlah_karakter', 'bag_of_words_top10', 'original_filename', 'extraction_method', 'text_full']


In [ ]:
verify_df = pd.read_csv(output_path)
assert len(verify_df) == len(metadata_df), "Jumlah baris tidak konsisten setelah disimpan!"
print(f"✅ Verifikasi berhasil: {len(verify_df)} baris terbaca kembali dengan benar dari {output_path.name}")
verify_df[["case_id", "no_perkara", "tanggal", "pihak", "jumlah_pasal_disebut"]].head(10)


✅ Verifikasi berhasil: 30 baris terbaca kembali dengan benar dari cases.csv


,case_id,no_perkara,tanggal,pihak,jumlah_pasal_disebut
0,case_001,1002/Pid.Sus/2022/PN Jkt.Tim,04 APRIL 2022,NOVALDI PRATOMO,10
1,case_002,1037/PID.SUS/2021/PN.JKT.Tim.,22 Maret 2022,A.H,4
2,case_003,11/Pid.Sus/2022/PN Jkt.Tim,29 Maret 2022,Jabbar,6
3,case_004,202/Pid.Sus/2023/PN. Jkt Tim,8 Januari 2024,HARIS AZHAR,10
4,case_005,203/Pid.Sus/2023/PN JKT.TIM,8 Januari 2024,FATIAH MAULIDIYANTY,10
5,case_006,212/Pid.Sus/2025/PN Jkt.Tim,NaN,TERDAKWA,8
6,case_007,330/Pid.Sus/2025/PN Jkt.Tim,14 Agustus 2025,Eko Candra Setiawan,7
7,case_008,351/Pid.Sus/2025/PN Jkt. Tim,NaN,IWAN MARDAN AKBAR BIN ALM,6
8,case_009,365/Pid.Sus/2023/PN JKT.Tim,19 September 2023,TAYO TOLU OMODARATAN alias ADAM,10
9,case_010,463/Pid.Sus/2024/PN Jkt.Tim,19 Desember 2024,KARMANSYAH LILI,10


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("../data/processed/cases.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {csv_path}")

df = pd.read_csv(csv_path)

print(f"Jumlah kasus: {len(df)}")
print(f"Jumlah kolom: {len(df.columns)}")

json_path = Path("../data/processed/cases.json")

df.to_json(
    json_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print(f"JSON berhasil disimpan di: {json_path}")

Jumlah kasus: 30
Jumlah kolom: 20
JSON berhasil disimpan di: ..\data\processed\cases.json


## 6. Catatan 
- Output Tahap 2 yang dihasilkan: `data/processed/cases.csv`

